# gambit_sim: overview

Simulation side of Gambit: the Gazebo world `gambit_chess` (SO-ARM101, board with ArUco corner
markers, 640 x 480 camera, 32 pieces + 2 spare queens per color), driven by teleporting static
piece models ("vanishing pieces").

## Deliverables (README "Updated plan (11/4/2026)")

| Plan item | Notebook(s) | Status |
|---|---|---|
| Gazebo world | `world_generator` (world from `board.yaml`, self-check vs `chess_world.sdf`) | world done; piece/board model generation TODO |
| Simulated calibration compared against the true intrinsics | `data_generator` (true K), `run_sim_calibration` | K done; ChArUco board in sim TODO |
| Pieces in Gazebo, PGN teleport replay script | `teleport`, `run_replay_game` | assignment + pose math done; ROS backend TODO |
| Homography and grid on simulated frames | `data_generator.board_corner_pixels` (exact corners to grade the homography) | done (ground truth); capture TODO |
| Gazebo vanishing pieces to test game logic | `teleport` (legal replay), `fault_injector`, `run_fault_tests` | fault generation done; inject/undo TODO |
| ResNet-18 sim only / mixed training data | `data_generator`, `run_generate_dataset` | manifest done; capture/crops TODO |

**Cut rule:** if the pieces don't render by November 11, Gazebo stays calibration-only: then only
`world_generator`, `data_generator` (intrinsics) and `run_sim_calibration` matter.

## Run order

1. `world_generator.ipynb` (self-check: the repo world matches board.yaml). Only re-write the world
   if you changed board.yaml.
2. Launch the sim: `ros2 launch gambit_bringup gambit.launch.py` (see `docs/gazebo.md` for the manual
   5-terminal version and gotchas; `../../docs/gazebo.md`).
3. `run_sim_calibration.ipynb`: simulated ChArUco calibration vs the true K.
4. `run_replay_game.ipynb`: teleport a PGN game ply by ply; watch it in Gazebo.
5. `run_generate_dataset.ipynb`: sim sessions -> `gambit_vision/data/crops/sim_*` (contracts.md).
6. `run_fault_tests.ipynb`: inject faults during replays, check the stack flags them.

Library notebooks (`%run` each other): `world_generator` -> `teleport` -> `data_generator` ->
`fault_injector`; `ros_playground` (rclpy notes). Configs: `gambit_bringup/config/board.yaml`,
`faults.yaml`, `configs/data_generator.yaml`. ROS nodes (C++) live in `gambit_ros/`.

## TODO checklist (package level)
- [ ] confirm the world on ROS 2 Lyrical + Gazebo Jetty (docs/gazebo.md was tested on Jazzy/Harmonic)
- [ ] teleport backend via ROS (`teleport.Ros2Backend`)
- [ ] frame capture + crops (reusing gambit_vision M2)
- [ ] ChArUco board model for simulated calibration
- [ ] fault tests with pass criteria

In [ ]:
from pathlib import Path
here = Path(".").resolve()
expected = ["world_generator", "teleport", "data_generator", "fault_injector", "ros_playground",
            "run_replay_game", "run_generate_dataset", "run_sim_calibration", "run_fault_tests"]
missing = [n for n in expected if not (here / f"{n}.ipynb").exists()]
print("missing notebooks:", missing or "none")

## Self-check

In [ ]:
assert not missing, missing
assert (here / "../../docs/gazebo.md").exists() and (here / "../../gambit_bringup/world/chess_world.sdf").exists()